In [1]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121

# 1. Install Unsloth from main (Nightly)
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. CRITICAL FIX: Force downgrade Transformers to 4.56.2
!pip install --upgrade --no-deps "transformers==4.56.2" "accelerate>=1.0.0" "peft>=0.12.0" "trl>=0.12.0"

# 3. Remove incompatible library
!pip uninstall torchao -y

In [2]:
import json
import sqlite3
import os

def schema_dict_to_create_sql(schema_dict):
    """
    Parses Spider 'tables.json' to generate accurate SQL DDL.
    Improvements:
    1. Uses ACTUAL column types (number -> INTEGER, text -> TEXT).
    2. Adds FOREIGN KEY constraints (Critical for JOINs).
    """
    try:
        # standard spider keys
        table_names = schema_dict.get("table_names_original", [])
        column_names = schema_dict.get("column_names_original", []) # [[table_idx, "col_name"], ...]
        column_types = schema_dict.get("column_types", []) # ["text", "number", ...]
        primary_keys = schema_dict.get("primary_keys", []) # [col_idx, ...]
        foreign_keys = schema_dict.get("foreign_keys", []) # [[src_col_idx, tgt_col_idx], ...]

        # Fallback for non-standard schemas
        if not table_names: 
            return str(schema_dict)

        # 1. Map Table IDs to Names
        table_map = {i: name for i, name in enumerate(table_names)}
        
        # 2. Organize Columns by Table
        # Structure: {table_id: [(col_idx, col_name, col_type), ...]}
        table_cols = {}
        for c_idx, (t_id, c_name) in enumerate(column_names):
            if t_id < 0: continue # Skip special '*' columns
            dtype = column_types[c_idx] if c_idx < len(column_types) else "text"
            table_cols.setdefault(t_id, []).append((c_idx, c_name, dtype))

        # 3. Build CREATE Statements
        create_stmts = []
        
        for t_id, cols in table_cols.items():
            t_name = table_map.get(t_id, f"table_{t_id}")
            definitions = []
            
            # -- Columns --
            for c_idx, c_name, c_type in cols:
                # Map Spider types to SQL types
                sql_type = "INTEGER" if c_type == "number" else "TEXT"
                def_str = f'"{c_name}" {sql_type}'
                
                # Add Primary Key inline (simplest for LLMs)
                if c_idx in primary_keys:
                    def_str += " PRIMARY KEY"
                
                definitions.append(def_str)
            
            # -- Foreign Keys --
            # Find FKs where the source column belongs to THIS table
            for src_idx, tgt_idx in foreign_keys:
                # check if src_idx is in the current table's columns
                if src_idx in [c[0] for c in cols]:
                    # Get target details
                    tgt_t_id = column_names[tgt_idx][0]
                    tgt_t_name = table_map.get(tgt_t_id, "unknown")
                    tgt_c_name = column_names[tgt_idx][1]
                    
                    # Get source column name
                    src_c_name = column_names[src_idx][1]
                    
                    fk_str = f"FOREIGN KEY (\"{src_c_name}\") REFERENCES {tgt_t_name}(\"{tgt_c_name}\")"
                    definitions.append(fk_str)

            stmt = f"CREATE TABLE {t_name} (\n  " + ",\n  ".join(definitions) + "\n);"
            create_stmts.append(stmt)

        return "\n\n".join(create_stmts)

    except Exception as e:
        print(f"Schema Parse Error: {e}")
        return ""

def execute_sql(sql, db_path):
    """Executes SQL and returns results (list of tuples) or Error string."""
    if not sql: return "Error: Empty SQL"
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(sql)
        result = cursor.fetchall()
        conn.close()
        return result
    except Exception as e:
        return f"Error: {e}"

In [3]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
import json
import os

# --- NUCLEAR FIX (Prevent Inference Crashes) ---
# We disable the compiler just like in training to be safe.
def no_op_compile(model, *args, **kwargs): return model
torch.compile = no_op_compile
torch._dynamo.config.disable = True
# -----------------------------------------------

# --- CONFIG ---
# ✅ POINTING TO YOUR SPECIFIC CHECKPOINT
CHECKPOINT_PATH = "/kaggle/input/qwen2-5-nl2sql-finetuned/outputs/checkpoint-2500"
SPIDER_PATH = "/kaggle/input/yale-universitys-spider-10-nlp-dataset/spider"

print(f"📂 Loading Fine-Tuned Adapter from: {CHECKPOINT_PATH}")

# 1. Load the Model + Adapters
# Unsloth automatically merges the LoRA adapters from your checkpoint
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = CHECKPOINT_PATH, 
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

# 2. Enable Inference Mode (2x Faster)
FastLanguageModel.for_inference(model)
tokenizer = get_chat_template(tokenizer, chat_template = "qwen-2.5")

# 3. Load Spider Data (Validation Set)
with open(f"{SPIDER_PATH}/dev.json", 'r') as f:
    dev_data = json.load(f)

# 4. Load & Parse Schemas
with open(f"{SPIDER_PATH}/tables.json", 'r') as f:
    tables_data = json.load(f)
    # Re-use your schema_dict_to_create_sql function from the previous cell
    schema_lookup = {t['db_id']: schema_dict_to_create_sql(t) for t in tables_data}

print(f"✅ Successfully loaded Ccomplete and {len(dev_data)} test examples.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-02-16 06:41:07.578956: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771224067.795225      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771224067.862706      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


[unsloth_zoo.log|WARNING]Unsloth: Could not patch trl.trainer.grpo_trainer: Direct module loading failed for UnslothGRPOTrainer: Unexpected optimization option triton.enable_persistent_tma_matmul, known options are ['TYPE_CHECKING', 'enable_auto_functionalized_v2', 'debug', 'disable_progress', 'verbose_progress', 'fx_graph_cache', 'fx_graph_remote_cache', 'autotune_local_cache', 'autotune_remote_cache', 'force_disable_caches', 'sleep_sec_TESTING_ONLY', 'custom_op_default_layout_constraint', 'cpp_wrapper', 'abi_compatible', 'c_shim_version', 'dce', 'static_weight_shapes', 'size_asserts', 'nan_asserts', 'pick_loop_orders', 'inplace_buffers', 'allow_buffer_reuse', 'memory_planning', 'memory_pool', 'benchmark_harness', 'epilogue_fusion', 'epilogue_fusion_first', 'pattern_matcher', 'b2b_gemm_pass', 'post_grad_custom_pre_pass', 'post_grad_custom_post_pass', 'joint_custom_pre_pass', 'joint_custom_post_pass', 'pre_grad_custom_pass', '_pre_fusion_custom_pass', 'split_cat_fx_passes', 'efficient_

📂 Loading Fine-Tuned Adapter from: /kaggle/input/qwen2-5-nl2sql-finetuned/outputs/checkpoint-2500
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ Successfully loaded Ccomplete and 1034 test examples.


In [4]:
import re
from tqdm import tqdm

def normalize_sql(sql):
    if not sql: return ""
    sql = sql.lower().strip().rstrip(";")
    sql = re.sub(r'\s+', ' ', sql)
    return sql

def is_set_match(gold_res, pred_res):
    if isinstance(gold_res, str) or isinstance(pred_res, str): return False
    if not gold_res or not pred_res: return False
    try:
        return set(gold_res) == set(pred_res)
    except:
        return False

# --- CONFIG ---
NUM_SAMPLES = len(dev_data) # Run on ALL examples
results = []

print(f"🚀 Starting Full Evaluation on {NUM_SAMPLES} samples...")

for i in tqdm(range(NUM_SAMPLES)):
    item = dev_data[i]
    db_id = item['db_id']
    question = item['question']
    gold_sql = item['query']
    
    # 1. Get Schema
    schema = schema_lookup.get(db_id, "")
    
    # 2. Prompting
    SYSTEM_PROMPT = (
    "You are an expert Text-to-SQL model. Generate a valid SQLite query to answer the user's question, "
    "using ONLY the tables and columns provided in the schema.\n"
    "Follow these strict rules:\n"
    "1. Output ONLY the raw SQL code. Do not use markdown format or explanations.\n"
    "2. Do not invent tables or columns that do not exist in the schema.\n"
    "3. Return columns in the EXACT order requested in the question.\n"
    "4. Do NOT join tables unless strictly necessary to connect data. Avoid redundant joins.\n"
    "5. Prefer 'ORDER BY ... LIMIT 1' over subqueries for finding maximums or top results.\n"
    "6. Apply all categorical filters (WHERE clauses) from the question before applying ordering or limits."
    )
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Schema:\n{schema}\n\nQuestion:\n{question}"},
    ]
    
    # 3. Generate
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        input_ids=inputs, 
        max_new_tokens=150, 
        use_cache=True, 
        temperature=0.0, 
        do_sample=False
    )
    pred_raw = tokenizer.batch_decode(outputs)[0]
    
    # 4. Extract SQL
    try:
        pred_sql = pred_raw.split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip()
        pred_sql = pred_sql.replace("```sql", "").replace("```", "").strip()
        if "\n" in pred_sql and not pred_sql.upper().startswith("SELECT"):
             lines = pred_sql.split('\n')
             pred_sql = lines[-1] 
    except:
        pred_sql = "ERROR_PARSING"

    # ============================================================
    # 🛠️ TOKENIZATION FIX (Added Here)
    # This repairs the split operators like "> =" into ">="
    # ============================================================
    pred_sql = re.sub(r'>\s+=', '>=', pred_sql)
    pred_sql = re.sub(r'<\s+=', '<=', pred_sql)
    pred_sql = re.sub(r'!\s+=', '!=', pred_sql)
    # ============================================================

    # 5. Execute & Compare
    db_file = f"{SPIDER_PATH}/database/{db_id}/{db_id}.sqlite"
    gold_res = execute_sql(gold_sql, db_file)
    pred_res = execute_sql(pred_sql, db_file)
    
    # Metrics
    ex_match = (str(gold_res) == str(pred_res)) and ("Error" not in str(pred_res))
    
    soft_match = False
    if not ex_match and "Error" not in str(pred_res):
        soft_match = is_set_match(gold_res, pred_res)
    
    # Save Result
    results.append({
        "db_id": db_id,
        "question": question,
        "gold": gold_sql,
        "pred": pred_sql,
        "ex_match": ex_match,
        "soft_match": soft_match
    })

# --- FINAL REPORT ---
strict_acc = sum(1 for r in results if r['ex_match']) / len(results)
relaxed_acc = sum(1 for r in results if r['ex_match'] or r['soft_match']) / len(results)

print(f"\n📊 FINAL RESULTS ({len(results)} samples):")
print(f"✅ Strict Accuracy:  {strict_acc:.2%}")
print(f"🤝 Relaxed Accuracy: {relaxed_acc:.2%}")

🚀 Starting Full Evaluation on 1034 samples...


100%|██████████| 1034/1034 [42:17<00:00,  2.45s/it]


📊 FINAL RESULTS (1034 samples):
✅ Strict Accuracy:  71.95%
🤝 Relaxed Accuracy: 75.24%


In [5]:
import pandas as pd

# Filter: Show ONLY cases where the data returned was wrong (Soft Match Failed)
# This ignores simple ordering differences and focuses on logic errors.
real_failures = [r for r in results if not r['soft_match'] and not r['ex_match']]

print(f"❌ Found {len(real_failures)} Logic/Syntax Failures (ignoring ordering differences).\n")
print("="*60)

# Loop through top failures
for i, fail in enumerate(real_failures[:10]): # Show top 10
    print(f"🔴 FAILURE CASE #{i+1} (DB: {fail['db_id']})")
    
    print(f"❓ Q: {fail['question']}")
    print(f"✅ GOLD: {fail['gold']}")
    print(f"❌ PRED: {fail['pred']}")
    
    print("="*60)
    print("\n")

❌ Found 256 Logic/Syntax Failures (ignoring ordering differences).

🔴 FAILURE CASE #1 (DB: concert_singer)
❓ Q: What are the names of the singers who performed in a concert in 2014?
✅ GOLD: SELECT T2.name FROM singer_in_concert AS T1 JOIN singer AS T2 ON T1.singer_id  =  T2.singer_id JOIN concert AS T3 ON T1.concert_id  =  T3.concert_id WHERE T3.year  =  2014
❌ PRED: SELECT T2.Name FROM singer_in_concert AS T1 JOIN singer AS T2 ON T1.Singer_ID   =   T2.Singer_ID JOIN concert AS T3 ON T2.Singer_ID   =   T3.concert_ID WHERE T3.Year   =   2014


🔴 FAILURE CASE #2 (DB: concert_singer)
❓ Q: Find the number of concerts happened in the stadium with the highest capacity.
✅ GOLD: SELECT count(*) FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id ORDER BY T2.Capacity DESC LIMIT 1
❌ PRED: SELECT count(*) FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T2.capacity   =    ( SELECT max(capacity) FROM stadium )


🔴 FAILURE CASE #3 (DB: concert_sing